In [1]:
"""
Speed (blue) and pressure (orange) snapshots for flow past a cylinder.

Uses exactly the same solver as the original cylinder code
(diffusion -> projection -> advection -> projection), but records
u, v AND p at chosen times, for Re = 25 (stable) and Re = 6000 (unstable).

Outputs
  speed_panels.png      speed |u|   (blue colormap)
  pressure_panels.png   pressure p  (orange/red colormap)

Pressure note
  The solver works with the scaled pressure  p_tilde = dt * p.
  We plot  p = p_tilde / dt,  in units of rho * U_in^2, with p = 0 at the outlet.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------------------------
# 1. Settings  (edit these to match the run you want)
# ------------------------------------------------
Nx, Ny = 280, 90            # use 300, 100 if that is the grid of your report figures
SX, SY = Nx + 2, Ny + 2

D = 14.0
cx = 40.0                   # cylinder centre x (cells)
cy = (Ny + 1) / 2.0         # cylinder centre y (cells)

U_in = 1.0
V_in = 0.0
EPS = 0.05
TOP_WALL_SPEED = 1.5 * U_in

dt = 0.5
JACOBI_ITERS = 150

RES = [25, 6000]                     # left column, right column
SNAP_TIMES = [25, 150, 300, 500]     # physical times t (t = step * dt)
N_STEPS = int(round(max(SNAP_TIMES) / dt))   # 1000 steps -> t = 500

# ------------------------------------------------
# 2. Grid and cylinder
# ------------------------------------------------
xx, yy = np.meshgrid(np.arange(SX), np.arange(SY), indexing="ij")
cylinder = (xx - cx) ** 2 + (yy - cy) ** 2 <= (D / 2) ** 2

ymid = (SY - 1) / 2.0
U_PROF = U_in * (1.0 + EPS * (np.arange(SY) - ymid) / (SY / 2.0))


# ------------------------------------------------
# 3. Boundary conditions
# ------------------------------------------------
def set_bnd_u(u):
    u[0, :] = U_PROF
    u[-1, :] = u[-2, :]
    u[:, 0] = 0.0
    u[:, -1] = TOP_WALL_SPEED


def set_bnd_v(v):
    v[0, :] = V_in
    v[-1, :] = v[-2, :]
    v[:, 0] = 0.0
    v[:, -1] = 0.0


def set_bnd_vel(u, v):
    set_bnd_u(u)
    set_bnd_v(v)
    u[cylinder] = 0.0
    v[cylinder] = 0.0


def set_bnd_pressure(p):
    p[0, :] = p[1, :]
    p[-1, :] = 0.0
    p[:, 0] = p[:, 1]
    p[:, -1] = p[:, -2]


# ------------------------------------------------
# 4. Diffusion, advection, projection (unchanged)
# ------------------------------------------------
def diffuse_component(x, x0, viscosity, dt, is_u):
    a = dt * viscosity
    c_inv = 1.0 / (1.0 + 4.0 * a)
    for _ in range(JACOBI_ITERS):
        x[1:-1, 1:-1] = (
            x0[1:-1, 1:-1]
            + a * (x[0:-2, 1:-1] + x[2:, 1:-1] + x[1:-1, 0:-2] + x[1:-1, 2:])
        ) * c_inv
        if is_u:
            set_bnd_u(x)
        else:
            set_bnd_v(x)
    return x


def advect(d, d0, u, v, dt, is_u):
    Xg, Yg = np.meshgrid(np.arange(1, Nx + 1), np.arange(1, Ny + 1), indexing="ij")
    x = np.clip(Xg - dt * u[1:-1, 1:-1], 0.5, Nx + 0.5)
    y = np.clip(Yg - dt * v[1:-1, 1:-1], 0.5, Ny + 0.5)
    i0 = x.astype(int); i1 = i0 + 1
    j0 = y.astype(int); j1 = j0 + 1
    s1 = x - i0; s0 = 1.0 - s1
    t1 = y - j0; t0 = 1.0 - t1
    d[1:-1, 1:-1] = (
        s0 * (t0 * d0[i0, j0] + t1 * d0[i0, j1])
        + s1 * (t0 * d0[i1, j0] + t1 * d0[i1, j1])
    )
    if is_u:
        set_bnd_u(d)
    else:
        set_bnd_v(d)
    return d


def project(u, v, p, div):
    div[1:-1, 1:-1] = -0.5 * (
        u[2:, 1:-1] - u[0:-2, 1:-1] + v[1:-1, 2:] - v[1:-1, 0:-2]
    )
    set_bnd_pressure(div)
    set_bnd_pressure(p)
    for _ in range(JACOBI_ITERS):
        p[1:-1, 1:-1] = (
            div[1:-1, 1:-1]
            + p[0:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, 0:-2] + p[1:-1, 2:]
        ) / 4.0
        set_bnd_pressure(p)
    u[1:-1, 1:-1] -= 0.5 * (p[2:, 1:-1] - p[0:-2, 1:-1])
    v[1:-1, 1:-1] -= 0.5 * (p[1:-1, 2:] - p[1:-1, 0:-2])
    set_bnd_vel(u, v)
    return u, v


# ------------------------------------------------
# 5. One simulation: record speed AND pressure
# ------------------------------------------------
def run_case(Re):
    viscosity = U_in * D / Re
    u = np.full((SX, SY), U_in)
    v = np.zeros((SX, SY))
    set_bnd_vel(u, v)
    p = np.zeros((SX, SY))
    div = np.zeros((SX, SY))

    snap_steps = {int(round(t / dt)): t for t in SNAP_TIMES}   # step count -> t
    speed_snaps, pres_snaps = {}, {}
    probe = []
    probe_x, probe_y = int(cx + 3 * D), int(round(cy))

    for n in range(1, N_STEPS + 1):
        u0, v0 = u.copy(), v.copy()
        diffuse_component(u, u0, viscosity, dt, True)
        diffuse_component(v, v0, viscosity, dt, False)
        project(u, v, p, div)

        u0, v0 = u.copy(), v.copy()
        advect(u, u0, u0, v0, dt, True)
        advect(v, v0, u0, v0, dt, False)
        project(u, v, p, div)          # p now belongs to the final, divergence-free u

        probe.append(v[probe_x, probe_y])

        if n in snap_steps:
            t = snap_steps[n]
            speed_snaps[t] = np.sqrt(u[1:-1, 1:-1] ** 2 + v[1:-1, 1:-1] ** 2)
            pres_snaps[t] = p[1:-1, 1:-1] / dt          # p_tilde -> p
    return speed_snaps, pres_snaps, np.array(probe)


def strouhal(probe):
    late = probe[len(probe) // 2:]
    late = late - late.mean()
    if np.abs(late).max() < 0.05:
        return None
    cr = np.where((late[:-1] < 0) & (late[1:] >= 0))[0]
    if len(cr) < 3:
        return None
    T = np.diff(cr).mean() * dt
    return (1.0 / T) * D / U_in


# ------------------------------------------------
# 6. Run both Reynolds numbers (results cached in sim_cache.npz)
# ------------------------------------------------
import os
CACHE = "sim_cache.npz"
results = {}
if os.path.exists(CACHE):
    print("Loading cached simulation from", CACHE)
    z = np.load(CACHE)
    for Re in RES:
        results[Re] = (
            {t: z[f"speed_{Re}_{t}"] for t in SNAP_TIMES},
            {t: z[f"pres_{Re}_{t}"] for t in SNAP_TIMES},
        )
else:
    store = {}
    for Re in RES:
        print(f"Running Re = {Re} ({N_STEPS} steps) ...", flush=True)
        sp, pr, probe = run_case(Re)
        results[Re] = (sp, pr)
        for t in SNAP_TIMES:
            store[f"speed_{Re}_{t}"] = sp[t]
            store[f"pres_{Re}_{t}"] = pr[t]
        St = strouhal(probe)
        print(f"  Strouhal number: {St if St is None else round(St, 3)}")
    np.savez(CACHE, **store)

# ------------------------------------------------
# 7. Plotting
# ------------------------------------------------
speed_cmap = plt.get_cmap("Blues_r").copy()      # dark = slow, light = fast (as in the report)
orange_cmap = LinearSegmentedColormap.from_list(
    "orange_pressure",
    ["#fff6e5", "#ffd9a0", "#ffa94d", "#f4721e", "#c93a12", "#7a1409"],
)

mask = cylinder[1:-1, 1:-1]

# shared colour limits for all panels (robust to start-up spikes)
all_p = np.concatenate([
    results[Re][1][t][~mask].ravel() for Re in RES for t in SNAP_TIMES
])
P_LO, P_HI = np.percentile(all_p, [1, 99])
print(f"Pressure colour range: {P_LO:.3f} to {P_HI:.3f}  (units of rho*U_in^2)")


def make_figure(kind, cmap, vmin, vmax, label, title, fname):
    fig, axes = plt.subplots(
        len(SNAP_TIMES), len(RES), figsize=(11, 8.2),
        sharex=True, sharey=True,
    )
    im = None
    for r, t in enumerate(SNAP_TIMES):
        for c, Re in enumerate(RES):
            ax = axes[r, c]
            field = results[Re][0 if kind == "speed" else 1][t].copy()
            im = ax.imshow(
                field.T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax,
                extent=[0, Nx, 0, Ny], interpolation="bilinear", aspect="auto",
            )
            ax.add_patch(plt.Circle((cx - 1, cy - 1), D / 2 + 0.6, facecolor="#2b2b2b",
                                    edgecolor="#cccccc", linewidth=0.8, zorder=3))
            name = "Stable" if Re == 25 else "Unstable"
            ax.set_title(f"{name}: Re = {Re},   t = {t}", fontsize=9)
            if c == 0:
                ax.set_ylabel("y (cells)", fontsize=8)
            if r == len(SNAP_TIMES) - 1:
                ax.set_xlabel("x (grid cells)", fontsize=8)
            ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=11)
    fig.subplots_adjust(left=0.07, right=0.88, top=0.92, bottom=0.07,
                        hspace=0.35, wspace=0.08)
    cax = fig.add_axes([0.91, 0.07, 0.02, 0.85])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=7)
    fig.savefig(fname, dpi=200, facecolor="white")
    plt.close(fig)
    print("Saved:", fname)


make_figure(
    "speed", speed_cmap, 0.0, 1.5,
    r"speed $|\mathbf{u}|$  (in units of $U_{in}$)",
    r"Speed of the flow around the circular obstacle (inflow speed $U_{in}=1$)",
    "speed_panels.png",
)
make_figure(
    "pressure", orange_cmap, P_LO, P_HI,
    r"pressure $p$  (in units of $\rho U_{in}^2$, $p=0$ at outlet)",
    r"Pressure of the flow around the circular obstacle (inflow speed $U_{in}=1$)",
    "pressure_panels.png",
)

Running Re = 25 (1000 steps) ...
  Strouhal number: None
Running Re = 6000 (1000 steps) ...
  Strouhal number: 0.183
Pressure colour range: -0.268 to 0.292  (units of rho*U_in^2)
Saved: speed_panels.png
Saved: pressure_panels.png
